In [ ]:
from spark_config import get_spark_session

spark = get_spark_session(app_name="ethereum_cleaning")

26/05/05 21:57:23 WARN Utils: Your hostname, khanhdo-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 192.168.118.128 instead (on interface ens33)
26/05/05 21:57:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 21:57:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import pyspark.sql.functions as F

## Xử lý file tokens:
Bỏ trùng address + bỏ token decimal > 18

In [3]:
# tokens.parquet
df_tokens = spark.read.parquet("/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/tokens.parquet")
df_tokens.printSchema()
df_tokens.show(5)
df_tokens.count()

root
 |-- address: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- decimals: string (nullable = true)
 |-- total_supply: string (nullable = true)



26/05/05 21:57:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------------------+------+-----+--------+--------------------+
|             address|symbol| name|decimals|        total_supply|
+--------------------+------+-----+--------+--------------------+
|0x4bfbff63d03d93a...|      |     |      18|10000000000000000...|
|0x3498eaf5ec296eb...|    NM|NenMo|       4|        880000000000|
|0x9d1a236f1de06ab...| SPEED|Speed|      18|24229371788186338...|
|0x15ff21740d9e52a...|  WARP| Warp|      18|22443493719574827...|
|0xd32e32780734440...|      |     |      18|10000000000000000...|
+--------------------+------+-----+--------+--------------------+
only showing top 5 rows



174014

In [5]:
# Dedup + lọc decimals null
df_tokens = df_tokens.dropDuplicates(["address"]).filter(F.col("decimals").isNotNull())

df_tokens.printSchema()
df_tokens.show(5)
df_tokens.count()

root
 |-- address: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- decimals: string (nullable = true)
 |-- total_supply: string (nullable = true)

+--------------------+------+--------------------+--------+--------------------+
|             address|symbol|                name|decimals|        total_supply|
+--------------------+------+--------------------+--------+--------------------+
|0x0000000000b3f87...|  GST2|         Gastoken.io|       2|             1361386|
|0x000000630a383f8...|  weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x000000d2870857f...|  weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x0000b6ab44789eb...|   VSN|      Vision Network|      18|           250000000|
|0x00016fab0fa144c...|  last|lite adaptable sy...|       1|           420000000|
+--------------------+------+--------------------+--------+--------------------+
only showing top 5 rows



174008

In [6]:

df_tokens_processed = df_tokens.filter(F.col("decimals") <= 18)
df_tokens_processed.distinct().show()

+--------------------+-------+--------------------+--------+--------------------+
|             address| symbol|                name|decimals|        total_supply|
+--------------------+-------+--------------------+--------+--------------------+
|0x0000000000b3f87...|   GST2|         Gastoken.io|       2|             1361386|
|0x000000630a383f8...|   weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x000000d2870857f...|   weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x0000b6ab44789eb...|    VSN|      Vision Network|      18|           250000000|
|0x00016fab0fa144c...|   last|lite adaptable sy...|       1|           420000000|
|0x00024378720f481...|    TSR|     TesraAiSuperNet|      18|10000000000000000...|
|0x0006abbe90dc7e6...|  IG-11| Mandalorian.Finance|      18|10000000000000000...|
|0x00075b94bbb96c5...| gnarco|              gnarco|       0|                1000|
|0x0014c8459a7c5c3...|    SOL|             Solarix|       8| 1000000000000000000|
|0x001634b3ba2c8

In [12]:
import pandas as pd
pdf = df_tokens_processed.toPandas()
pdf.to_csv("/home/khanhdo/Documents/project/bigdata_mining/data_processed/processed_tokens.csv", index=False)

In [26]:
file_token_transfers_paths = [
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/token_transfers_t4_1.parquet",
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/token_transfers_t4_2.parquet",
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/token_transfers_t4_3.parquet"
]
df_token_transfers = spark.read.parquet(*file_token_transfers_paths)
df_token_transfers.printSchema()

df_token_transfers.show(5)
df_token_transfers.count()

root
 |-- token_address: string (nullable = true)
 |-- from_address: string (nullable = true)
 |-- to_address: string (nullable = true)
 |-- value: string (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)

+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|       token_address|        from_address|          to_address|               value|    transaction_hash|    block_timestamp|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|0x56072c95faa7012...|0xbdb3ba9ffe39254...|0x2621cc0b3f3c079...|81613886276036476...|0xfa972e55c437b46...|2026-04-07 13:20:59|
|0xcf58c682497f480...|0x3c308ebda6d205b...|0xfd820f61300ea9f...|         76000000000|0x0fd9f3723ddf2b5...|2026-04-07 13:29:23|
|0x82550f69042b498...|0x7a2bb38c73506b3...|0xef635552d32281a...|  127298014489419000|0x2a7671

89782413

In [28]:
# Cast the 'value' column to decimal(38, 0)
df_token_transfers = df_token_transfers.withColumn("value", F.expr("try_cast(value as decimal(38,0))"))
df_token_transfers.printSchema()
df_token_transfers.show(5)

root
 |-- token_address: string (nullable = true)
 |-- from_address: string (nullable = true)
 |-- to_address: string (nullable = true)
 |-- value: decimal(38,0) (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)

+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|       token_address|        from_address|          to_address|               value|    transaction_hash|    block_timestamp|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|0x56072c95faa7012...|0xbdb3ba9ffe39254...|0x2621cc0b3f3c079...|81613886276036476...|0xfa972e55c437b46...|2026-04-07 13:20:59|
|0xcf58c682497f480...|0x3c308ebda6d205b...|0xfd820f61300ea9f...|         76000000000|0x0fd9f3723ddf2b5...|2026-04-07 13:29:23|
|0x82550f69042b498...|0x7a2bb38c73506b3...|0xef635552d32281a...|  127298014489419000|0

### Join df_token_transfers vs df_tokens_processed

In [29]:
df_joined = df_token_transfers.join(
    df_tokens_processed,
    df_token_transfers["token_address"] == df_tokens_processed["address"], how = 'inner')
df_joined.show(20)
count = df_joined.count()
print(f"Số lượng bản ghi sau khi join: {count}")

+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+------+--------------------+--------+--------------------+
|       token_address|        from_address|          to_address|               value|    transaction_hash|    block_timestamp|             address|symbol|                name|decimals|        total_supply|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+------+--------------------+--------+--------------------+
|0xf629cbd94d3791c...|0x105757d992c3107...|0xe56c60b5f9f7b5f...|71381530726287540224|0xf638c0c1335df5c...|2026-04-11 06:25:23|0xf629cbd94d3791c...|   ENJ|          Enjin Coin|      18|10000000000000000...|
|0x940a2db1b7008b6...|0xdfd5293d8e347df...|0xfe2631026829332...|34962383557700000...|0x58d125b28e08104...|2026-04-02 17:03:23|0x940a2db1b7008b6...|  DUSK|        Dusk Network| 

Số lượng bản ghi sau khi join: 36281175


In [30]:
df_joined_processed = df_joined.withColumn(
    "adjusted_value",
    F.col("value") / (10 ** F.col("decimals"))
).select(
    "transaction_hash",
    "from_address",
    "to_address",
    "adjusted_value",
    "block_timestamp",
    "token_address",
    "name",
    "symbol",
    "decimals"
)
df_joined_processed.show(10)

+--------------------+--------------------+--------------------+------------------+-------------------+--------------------+--------------------+------+--------+
|    transaction_hash|        from_address|          to_address|    adjusted_value|    block_timestamp|       token_address|                name|symbol|decimals|
+--------------------+--------------------+--------------------+------------------+-------------------+--------------------+--------------------+------+--------+
|0xf638c0c1335df5c...|0x105757d992c3107...|0xe56c60b5f9f7b5f...| 71.38153072628754|2026-04-11 06:25:23|0xf629cbd94d3791c...|          Enjin Coin|   ENJ|      18|
|0x58d125b28e08104...|0xdfd5293d8e347df...|0xfe2631026829332...|3496.2383557699995|2026-04-02 17:03:23|0x940a2db1b7008b6...|        Dusk Network|  DUSK|      18|
|0xf51676d77141ece...|0x9008d19f58aabd9...|0x4d551bd8f20c899...| 54.78305758473262|2026-04-02 18:13:59|0x6810e776880c029...|        Gnosis Token|   GNO|      18|
|0x39a4d36b8b3fe4e...|0x1f69

In [31]:
df_token_transfers_filtered = df_joined_processed.select(
    "from_address",
    "to_address",
    "token_address",
    "adjusted_value",
    "block_timestamp"
)

In [32]:
df_token_transfers_filtered.write.mode("append").parquet("/home/khanhdo/Documents/project/bigdata_mining/data_processed/processed_token_transfers")

In [33]:
df_token_transfers_filtered = spark.read.parquet("/home/khanhdo/Documents/project/bigdata_mining/data_processed/processed_token_transfers")

In [34]:
df_token_transfers_filtered.count()

103583175

In [35]:
# Tao 2 luong token ra va vao
df_inflow = df_token_transfers_filtered.select(
    F.lower(F.col("to_address")).alias("user_address"),
    F.lower(F.col("token_address")).alias("token_address"),
    F.col("adjusted_value").cast("double").alias("value"),
    F.col("block_timestamp")
)

df_outflow = df_token_transfers_filtered.select(
    F.lower(F.col("from_address")).alias("user_address"),
    F.lower(F.col("token_address")).alias("token_address"),
    (F.col("adjusted_value").cast("double") * F.lit(-1.0)).alias("value"),
    F.col("block_timestamp")
)

df_inflow.show(5, truncate=False)
df_outflow.show(5, truncate=False)

+------------------------------------------+------------------------------------------+------------+-------------------+
|user_address                              |token_address                             |value       |block_timestamp    |
+------------------------------------------+------------------------------------------+------------+-------------------+
|0xb39a41e70b4a590e4f367339852a3233880eee9c|0xdac17f958d2ee523a2206206994597c13d831ec7|0.001       |2026-03-22 06:25:23|
|0x3ddc989d7a09b93b90701275cbf55087c3f04b01|0xdac17f958d2ee523a2206206994597c13d831ec7|50.014149   |2026-03-25 05:45:59|
|0x27c9a1caf417f31725a217998b4c265eb02f91d3|0xdac17f958d2ee523a2206206994597c13d831ec7|1.0E-5      |2026-03-25 05:53:23|
|0xf5f00413802fdb2009fdc64734e7ddbc37b79e49|0xdac17f958d2ee523a2206206994597c13d831ec7|0.001       |2026-03-25 06:06:35|
|0x9642b23ed1e01df1092b92641051881a322f5d4e|0xdac17f958d2ee523a2206206994597c13d831ec7|16253.460112|2026-03-25 06:13:11|
+-------------------------------

### Tính balance bằng tổng giá trị giao dịch với (vào: value +, ra: value - ), tổng số lần giao dịch mỗi token và lần giao dịch gần nhất

In [36]:
df_user_token_flow = df_inflow.union(df_outflow)

df_portfolios = (
    df_user_token_flow
    .groupBy("user_address", "token_address")
    .agg(
        F.sum("value").alias("balance"),
        F.count("*").alias("tx_count"), # Số lượng giao dịch liên quan đến token này
        F.max("block_timestamp").alias("last_active")
    )
    .filter(F.col("balance") > 0)
    .filter(F.col("user_address").isNotNull())
)

df_portfolios.show(20, truncate=False)
print("Portfolio rows:", df_portfolios.count())

+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+
|user_address                              |token_address                             |balance              |tx_count|last_active        |
+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+
|0xbbbbbbbbbb9cc5e90e3b3af64bdaf62c37eeffcb|0xdac17f958d2ee523a2206206994597c13d831ec7|1.6596641069574323E8 |147044  |2026-05-01 06:58:59|
|0x16a9be9be7f39c0a48a90e736990c20ce5a8c601|0xdac17f958d2ee523a2206206994597c13d831ec7|425.35603600012837   |488     |2026-04-03 19:47:11|
|0x6edfdda879466c4d146f309c2956189cf40c511b|0xdac17f958d2ee523a2206206994597c13d831ec7|0.0023199999995995313|11      |2026-04-02 02:58:47|
|0xd5610d2deb73ff788e6b20d99c68b10bc18bc19b|0xdac17f958d2ee523a2206206994597c13d831ec7|133.756464           |1       |2026-03-31 23:50:11|
|0xe189f12c67287adca77d9db0

Portfolio rows: 3515585


## Gán nhãn is_contract để phân biệt contract vs EOA

In [37]:
contracts_paths = [
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/contracts_1.parquet",
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/contracts_2.parquet"
]

df_contracts = (
    spark.read.parquet(*contracts_paths)
    .select(F.lower(F.col("address")).alias("address"))
    .dropDuplicates(["address"])
)

df_portfolios_labeled = (
    df_portfolios.alias("p")
    .join(df_contracts.alias("c"), F.col("p.user_address") == F.col("c.address"), "left")
    .withColumn(
        "is_contract",
        F.when(F.col("c.address").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )
    .select(
        F.col("p.user_address").alias("user_address"),
        F.col("p.token_address").alias("token_address"),
        F.col("p.balance").alias("balance"),
        F.col("p.tx_count").alias("tx_count"),
        F.col("p.last_active").alias("last_active"),
        F.col("is_contract")
    )
)

df_portfolios_labeled.show(20, truncate=False)
print("Total portfolios:", df_portfolios_labeled.count())
print("Contracts:", df_portfolios_labeled.filter(F.col("is_contract") == True).count())
print("EOA users:", df_portfolios_labeled.filter(F.col("is_contract") == False).count())

26/05/05 22:48:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:48:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:48:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:48:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:48:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:48:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+-----------+
|user_address                              |token_address                             |balance              |tx_count|last_active        |is_contract|
+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+-----------+
|0x031495920177ea9455d95c8a0a2bb3f592fa6c16|0xdac17f958d2ee523a2206206994597c13d831ec7|1.0300000000003015E-4|479     |2026-03-28 12:58:59|false      |
|0x149109613f179eddc6df2916d757ac465d93c14a|0xdac17f958d2ee523a2206206994597c13d831ec7|269.7035009999963    |44      |2026-04-02 04:15:59|false      |
|0x5d6cc3e967e19d0173ffd034292808046362d1ef|0xdac17f958d2ee523a2206206994597c13d831ec7|0.012782000000000002 |36      |2026-03-26 06:52:11|false      |
|0x69355223a0ce30aee41d353387c3082e5aafc4da|0xdac17f958d2ee523a2206206994597c13d831ec7|9.99870

Total portfolios: 3515585


26/05/05 22:50:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:50:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:50:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:50:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:51:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:51:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Contracts: 84962


26/05/05 22:52:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:52:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:52:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:52:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:52:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 22:53:02 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


EOA users: 3430623


## Lấy danh sách user có tổng giao dịch < 5000,  > 5000 khả năng lớn là ví sàn

In [38]:
# Danh sach user EOA co tong giao dich < 5000 
df_real_users = (
    df_portfolios_labeled
    .filter(F.col("is_contract") == False)
    .groupBy("user_address")
    .agg(F.sum("tx_count").alias("total_tx_count"))
    .filter(F.col("total_tx_count") < 5000)
)

df_real_user_portfolios = (
    df_portfolios_labeled
    .filter(F.col("is_contract") == False)
    .join(df_real_users.select("user_address"), on="user_address", how="inner")
)

df_real_user_portfolios.printSchema()
df_real_user_portfolios.show(20, truncate=False)
print("Rows:", df_real_user_portfolios.count())

root
 |-- user_address: string (nullable = true)
 |-- token_address: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- tx_count: long (nullable = false)
 |-- last_active: timestamp (nullable = true)
 |-- is_contract: boolean (nullable = false)



26/05/05 23:02:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:02:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:02:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:02:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:02:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+-----------+
|user_address                              |token_address                             |balance               |tx_count|last_active        |is_contract|
+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+-----------+
|0x0000401d0446ed6c6140d3dd106b42a7845c51d8|0xdac17f958d2ee523a2206206994597c13d831ec7|0.0025680000000000004 |5       |2026-03-01 11:24:35|false      |
|0x00020d627c1c182d82bd7dcc7925c295af2e02a7|0xdac17f958d2ee523a2206206994597c13d831ec7|14.51                 |1       |2026-03-05 05:55:35|false      |
|0x0002f2b93577e26bf405e6d41588facfc0f9cf50|0xdac17f958d2ee523a2206206994597c13d831ec7|2.710505431213761E-20 |4       |2026-04-09 04:18:59|false      |
|0x000489178e164610cb7755e54a541be103023e63|0xdac17f958d2ee523a2206206994597c13d831ec7|0

26/05/05 23:04:16 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:04:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:04:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:04:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:04:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:04:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Rows: 3427540


In [39]:
output_dir = "/home/khanhdo/Documents/project/bigdata_mining/data_processed/real_user_portfolios"
df_real_user_portfolios.write.mode("overwrite").parquet(output_dir)
print(f"Saved Spark parquet dataset to: {output_dir}")

26/05/05 23:09:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:09:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:09:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:09:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:09:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/05 23:09:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Saved Spark parquet dataset to: /home/khanhdo/Documents/project/bigdata_mining/data_processed/real_user_portfolios
